In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from cubic_hc import F, predict, correct, solve

In [ ]:
# Set random seed for reproducibility
np.random.seed(2026)

# Coefficients for F(x) = x^3 + ax^2 + bx + c (i.e., x^3 - 4*x^2 + 8*x - 8)
a, b, c = -4, 8, -8

sols, paths, gamma = solve(a, b, c, dt=1e-3)

for root in sols:
    assert np.abs(F(root, a, b, c)) < 1e-10, f"F({root}) = {F(root, a, b, c)}"

In [ ]:
# Plot the homotopy continuation paths
colors = ["tab:blue", "tab:orange", "tab:green"]
fig, ax = plt.subplots(figsize=(4, 4))
all_x = np.concatenate([p[0] for p in paths])
cx = (all_x.real.max() + all_x.real.min()) / 2
cy = (all_x.imag.max() + all_x.imag.min()) / 2

for i, (path_x, path_t) in enumerate(paths):
    ax.plot(path_x.real, path_x.imag, color=colors[i])
    ax.scatter(path_x[0].real, path_x[0].imag, color=colors[i], marker="o", s=40)
    ax.scatter(path_x[-1].real, path_x[-1].imag, color=colors[i], marker="X", s=40)

# Legend for markers only
ax.scatter([], [], color="gray", marker="o", s=40, label="Start")
ax.scatter([], [], color="gray", marker="X", s=40, label="End")
ax.legend()
ax.set_xlabel("Real axis")
ax.set_ylabel("Imaginary axis")

# Make square: same range for both axes
margin = 0.1
max_range = max(
    all_x.real.max() - all_x.real.min(), all_x.imag.max() - all_x.imag.min()
)
half = max_range / 2 + margin
ax.set_xlim(cx - half, cx + half)
ax.set_ylim(cy - half, cy + half)
ax.axhline(0, color="gray", linewidth=0.5, alpha=0.5)
ax.axvline(0, color="gray", linewidth=0.5, alpha=0.5)

# save figure as png
plt.savefig("hc_paths.png", dpi=300, bbox_inches="tight")

In [ ]:
# Illustrate one predictor-corrector step
path_x, path_t = paths[2]
index = int(len(path_x) * 0.3)
x_t = path_x[index]
t = path_t[index]
dt = 2e-1

x_t_next = predict(x=x_t, t=t, dt=dt, gamma=gamma, a=a, b=b, c=c)
_, corrector_path = correct(x=x_t_next, t=t+dt, gamma=gamma, a=a, b=b, c=c)
corrector_path = np.array(corrector_path)

fig, ax = plt.subplots(figsize=(4, 4))
ax.plot(path_x.real[200:600], path_x.imag[200:600], color="tab:blue")
ax.plot([x_t.real, x_t_next.real], [x_t.imag, x_t_next.imag], linestyle="dotted", color="tab:orange")
ax.plot(corrector_path.real, corrector_path.imag, linestyle="dotted", color="tab:blue")
ax.scatter([x_t.real], [x_t.imag], marker="o", s=50, color="tab:blue")
ax.scatter([x_t_next.real], [x_t_next.imag], marker="o", s=50, color="tab:orange")
ax.scatter([corrector_path[-1].real], [corrector_path[-1].imag], marker="o", s=50, color="tab:blue")
ax.set_xlabel("Real axis")
ax.set_ylabel("Imaginary axis")

# save figure
plt.savefig("hc_predictor_corrector.png", dpi=300, bbox_inches="tight")